# 18 — K-Prototypes: clusterização nativa para dados mistos
## PRT Seguros

O K-Means original trata as colunas categóricas (já em one-hot) como se fossem números contínuos —
tecnicamente funciona, mas a distância euclidiana entre `segmento_Ouro=1` e `segmento_Ouro=0` não
tem o mesmo significado que a distância entre duas idades, por exemplo.

O **K-Prototypes** (biblioteca `kmodes`) resolve isso combinando duas métricas de distância na mesma
clusterização: distância euclidiana para as colunas numéricas + *matching dissimilarity* (conta
quantas categorias diferem) para as colunas categóricas. Testamos se o cluster resultante é mais
informativo que o do K-Means.

## 1. Carregar dados e preparar colunas numéricas + categóricas (sem one-hot)

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready.csv")

GRUPOS_CATEGORICOS = {
    "escolaridade": ["escolaridade_Fundamental", "escolaridade_Medio", "escolaridade_NaN",
                      "escolaridade_Pos", "escolaridade_Superior"],
    "canal_aquisicao": ["canal_aquisicao_Agente", "canal_aquisicao_Digital", "canal_aquisicao_Indicacao",
                         "canal_aquisicao_NaN", "canal_aquisicao_Telefone"],
    "canal": ["canal_Carta", "canal_Email", "canal_Nao Informado", "canal_Telefone", "canal_WhatsApp"],
    "metodo_pagamento": ["metodo_pagamento_NaN", "metodo_pagamento_boleto", "metodo_pagamento_credito",
                          "metodo_pagamento_debito", "metodo_pagamento_pix"],
    "tipo_cobertura": ["tipo_cobertura_NaN", "tipo_cobertura_basica", "tipo_cobertura_padrao",
                        "tipo_cobertura_premium"],
    "segmento": ["segmento_Bronze", "segmento_Diamante", "segmento_NaN", "segmento_Ouro", "segmento_Prata"],
    "veiculo": ["veiculo_Carro", "veiculo_Moto", "veiculo_NaN", "veiculo_Pickup", "veiculo_SUV", "veiculo_Van"],
    "regiao": ["regiao_Centro-Oeste", "regiao_NaN", "regiao_Nordeste", "regiao_Sudeste", "regiao_Sul"],
}

def reconstruir_categoricas(df):
    df = df.copy()
    for grupo, cols in GRUPOS_CATEGORICOS.items():
        prefixo = grupo + "_"
        df[grupo] = df[cols].idxmax(axis=1).str.replace(prefixo, "", regex=False)
        df = df.drop(columns=cols)
    return df

train_r = reconstruir_categoricas(train.drop(columns=["cluster"]))
val_r = reconstruir_categoricas(val.drop(columns=["cluster"]))
kaggle_r = reconstruir_categoricas(kaggle.drop(columns=["cluster"]))

CATEGORICAS_KPROTO = list(GRUPOS_CATEGORICOS.keys()) + [
    "genero", "estado_civil", "tem_filhos", "pagamento_em_dia", "possui_imovel", "indicou_clientes"
]
NUMERICAS_KPROTO = [
    "tempo_cliente_dias", "num_apolices_ativas", "num_produtos_contratados",
    "desconto_aplicado_pct", "indice_relacionamento", "satisfacao_nps",
    "renda_anual", "qtd_dependentes", "valor_imovel", "valor_cobertura_total", "idade",
    "score_engajamento_digital", "num_reclamacoes_12m", "num_acessos_app_mes", "franquia_media",
]
COLS_KPROTO = NUMERICAS_KPROTO + CATEGORICAS_KPROTO
print(f"Numéricas: {len(NUMERICAS_KPROTO)} | Categóricas: {len(CATEGORICAS_KPROTO)}")


Numéricas: 15 | Categóricas: 14


## 2. Padronizar numéricas e ajustar o K-Prototypes numa amostra (fit rápido) + prever em todo mundo

`kmodes` é uma implementação pura em Python — bem mais lenta que o K-Means do sklearn. Ajustamos
os protótipos numa amostra de 30 mil linhas (treino + Kaggle, sem rótulo) e depois usamos
`.predict()` nas bases inteiras — os centróides não mudam, só a atribuição de cada linha.

In [2]:
imp_num = SimpleImputer(strategy="median")
scaler_num = StandardScaler()

pool_fit = pd.concat([train_r[COLS_KPROTO], kaggle_r[COLS_KPROTO]], ignore_index=True)
pool_fit[NUMERICAS_KPROTO] = imp_num.fit_transform(pool_fit[NUMERICAS_KPROTO])
pool_fit[NUMERICAS_KPROTO] = scaler_num.fit_transform(pool_fit[NUMERICAS_KPROTO])
for c in CATEGORICAS_KPROTO:
    pool_fit[c] = pool_fit[c].astype(str)

amostra = pool_fit.sample(n=30_000, random_state=RANDOM_STATE)
idx_cat = [amostra.columns.get_loc(c) for c in CATEGORICAS_KPROTO]

kproto = KPrototypes(n_clusters=6, init="Huang", n_init=2, random_state=RANDOM_STATE, verbose=0)
kproto.fit(amostra.values, categorical=idx_cat)
print("K-Prototypes ajustado na amostra de 30k linhas.")


K-Prototypes ajustado na amostra de 30k linhas.


In [3]:
def preparar_para_kproto(df):
    saida = df[COLS_KPROTO].copy()
    saida[NUMERICAS_KPROTO] = imp_num.transform(saida[NUMERICAS_KPROTO])
    saida[NUMERICAS_KPROTO] = scaler_num.transform(saida[NUMERICAS_KPROTO])
    for c in CATEGORICAS_KPROTO:
        saida[c] = saida[c].astype(str)
    return saida

train_kp = preparar_para_kproto(train_r)
val_kp = preparar_para_kproto(val_r)
kaggle_kp = preparar_para_kproto(kaggle_r)

train_r["cluster_kproto"] = kproto.predict(train_kp.values, categorical=idx_cat)
val_r["cluster_kproto"] = kproto.predict(val_kp.values, categorical=idx_cat)
kaggle_r["cluster_kproto"] = kproto.predict(kaggle_kp.values, categorical=idx_cat)

churn_por_cluster_kp = train_r.groupby("cluster_kproto")[TARGET_COL].agg(["mean", "count"])
print(churn_por_cluster_kp.sort_values("mean", ascending=False))


                    mean  count
cluster_kproto                 
3               0.287307  13637
4               0.189960  16135
0               0.084302  12823
5               0.077341  12529
2               0.036339  10237
1               0.018444  14639


## 3. Treinar CatBoost (categóricas nativas) trocando `cluster` (K-Means) por `cluster_kproto`

In [4]:
train_r["cluster_kproto"] = train_r["cluster_kproto"].astype(str)
val_r["cluster_kproto"] = val_r["cluster_kproto"].astype(str)
kaggle_r["cluster_kproto"] = kaggle_r["cluster_kproto"].astype(str)

COLUNAS_CATEGORICAS_FINAL = list(GRUPOS_CATEGORICOS.keys()) + ["cluster_kproto"]
feature_cols = [c for c in train_r.columns if c not in (ID_COL, TARGET_COL)]
cat_idx_final = [feature_cols.index(c) for c in COLUNAS_CATEGORICAS_FINAL]

X_train, y_train = train_r[feature_cols], train_r[TARGET_COL]
X_val, y_val = val_r[feature_cols], val_r[TARGET_COL]
X_kaggle = kaggle_r[feature_cols]

X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

pool_tr = Pool(X_tr_es, y_tr_es, cat_features=cat_idx_final)
pool_es = Pool(X_es, y_es, cat_features=cat_idx_final)
pool_val = Pool(X_val, cat_features=cat_idx_final)
pool_kaggle = Pool(X_kaggle, cat_features=cat_idx_final)

cat_kproto = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
)
cat_kproto.fit(pool_tr, eval_set=pool_es)

proba_val_kproto = cat_kproto.predict_proba(pool_val)[:, 1]
auc_kproto = roc_auc_score(y_val, proba_val_kproto)
print(f"CatBoost + cluster K-Prototypes: AUC-ROC val = {auc_kproto:.4f}")
print(f"CatBoost + cluster K-Means (ref, notebook 16): AUC-ROC val = 0.8264")


CatBoost + cluster K-Prototypes: AUC-ROC val = 0.8268
CatBoost + cluster K-Means (ref, notebook 16): AUC-ROC val = 0.8264


## 4. Gerar submissão (só se o K-Prototypes ganhar do K-Means)

In [5]:
if auc_kproto >= 0.8264:
    proba_kaggle_kproto = cat_kproto.predict_proba(pool_kaggle)[:, 1]
    submissao = pd.DataFrame({"Id": kaggle_r[ID_COL], "probabilidade_churn": proba_kaggle_kproto})
    submissao.to_csv("submissions/submission_kprototypes.csv", index=False)
    print(f"Melhorou! Salvo: submissions/submission_kprototypes.csv (AUC-ROC val = {auc_kproto:.4f})")
else:
    print(f"K-Prototypes não superou o K-Means na validação ({auc_kproto:.4f} vs 0.8264) — não gerou submissão.")


Melhorou! Salvo: submissions/submission_kprototypes.csv (AUC-ROC val = 0.8268)
